In [1]:
import sys
print(sys.executable)

C:\Users\mitch\source\repos\mtg_ai\.venv\Scripts\python.exe


In [3]:
# Imports & display config

import pandas as pd
from pathlib import Path

# Make pandas usable for wide MTG data
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 0)

PROJECT_ROOT = Path("..").resolve()
DATA = PROJECT_ROOT / "data" / "processed"

In [5]:
# Load datasets

cards_core = pd.read_parquet(DATA / "cards_core.parquet")
cards_enriched = pd.read_parquet(DATA / "cards_enriched.parquet")
type_features = pd.read_parquet(DATA / "type_features.parquet")
oracle_tokens = pd.read_parquet(DATA / "oracle_tokens.parquet")

print("Loaded:")
print("cards_core:", cards_core.shape)
print("cards_enriched:", cards_enriched.shape)
print("type_features:", type_features.shape)
print("oracle_tokens:", oracle_tokens.shape)

Loaded:
cards_core: (36708, 19)
cards_enriched: (36708, 23)
type_features: (36708, 8)
oracle_tokens: (36708, 5)


In [15]:
# Clean head() views (select columns)

cards_enriched[
    [
        "name",
        "type_line",
        "basic_types",
        "sub_types",
        "cmc",
        "oracle_text_norm",
    ]
].head(10)

In [17]:
# Inspect tokenization quality

oracle_tokens[
    ["name", "oracle_text_norm", "oracle_tokens"]
].sample(5, random_state=42)

,name,oracle_text_norm,oracle_tokens
20917,Caravan Vigil,"Search your library for a basic land card, reveal it, put it into your hand, then shuffle.\nMorbid — You may put that card onto the battlefield instead of putting it into your hand if a creature died this turn.","[search, your, library, for, a, basic, land, card, reveal, it, put, it, into, your, hand, then, shuffle, NEWLINE, morbid, —, you_may, put, that, card, onto, the, battlefield, instead, of, putting, it, into, your, hand, if, a, creature, died, this, turn]"
18857,Bishop's Soldier,Lifelink (Damage dealt by this creature also causes you to gain that much life.),"[lifelink, (damage, dealt, by, this, creature, also, causes, you, to, gain, that, much, life, )]"
27507,Raging Kronch,This creature can't attack alone.,"[this, creature, can't, attack, alone]"
35527,"Gonti, Night Minister // Gonti, Night Minister",,[]
29950,Dragon-Style Twins,"Double strike\nProwess (Whenever you cast a noncreature spell, this creature gets +1/+1 until end of turn.)","[double, strike, NEWLINE, prowess, (whenever, you, cast, a, noncreature, spell, this, creature, gets, +1/+1, until_end_of_turn, )]"


In [19]:
# Look at a single known card

oracle_tokens[oracle_tokens["name"] == "Lightning Bolt"][
    ["oracle_text_norm", "oracle_tokens"]
]

,oracle_text_norm,oracle_tokens
9748,this card deals 3 damage to any target.,"[this, card, deals, NUM:3, damage, to, any, target]"


In [21]:
# Inspect type parsing edge cases

type_features.sample(10, random_state=1)[
    ["name", "type_line", "basic_types", "super_types", "sub_types"]
]

,name,type_line,basic_types,super_types,sub_types
18046,Working Stiff,Creature — Mummy,[Creature],[],[Mummy]
2363,"Appa, Aang's Companion // Appa, Aang's Companion",Card // Card,[],"[Card, Card]",[]
15175,Flight,Enchantment — Aura,[Enchantment],[],[Aura]
8632,Fleshtaker // Fleshtaker,Card // Card,[],"[Card, Card]",[]
22145,Drone,Token Artifact Creature — Drone,"[Artifact, Creature]",[Token],[Drone]
13942,Nekrataal Avatar,Vanguard,[],[Vanguard],[]
12824,Chandra's Spitfire,Creature — Elemental,[Creature],[],[Elemental]
15114,Guise of Fire,Enchantment — Aura,[Enchantment],[],[Aura]
32487,Traumatize,Sorcery,[Sorcery],[],[]
8621,Auratouched Mage,Creature — Human Wizard,[Creature],[],"[Human, Wizard]"


In [25]:
# Test problem children

type_features[type_features["name"].isin([
    "Dryad Arbor",
    "Urza, Lord High Artificer",
    "Fire // Ice",
    "Westvale Abbey"
])][
    ["name", "type_line", "basic_types", "super_types", "sub_types", "type_faces"]
]

,name,type_line,basic_types,super_types,sub_types,type_faces
24929,Fire // Ice,Instant // Instant,"[Instant, Instant]",[],[],"[{'basic_types': ['Instant'], 'raw_face_type_line': 'Instant', 'sub_types': [], 'super_types': []}, {'basic_types': ['Instant'], 'raw_face_type_line': 'Instant', 'sub_types': [], 'super_types': []}]"
33284,"Urza, Lord High Artificer",Legendary Creature — Human Artificer,[Creature],[Legendary],"[Human, Artificer]","[{'basic_types': ['Creature'], 'raw_face_type_line': 'Legendary Creature — Human Artificer', 'sub_types': ['Human', 'Artificer'], 'super_types': ['Legendary']}]"
33439,Dryad Arbor,Land Creature — Forest Dryad,"[Land, Creature]",[],"[Forest, Dryad]","[{'basic_types': ['Land', 'Creature'], 'raw_face_type_line': 'Land Creature — Forest Dryad', 'sub_types': ['Forest', 'Dryad'], 'super_types': []}]"


In [27]:
# Merge clusters for inspection

clusters = pd.read_parquet(DATA / "oracle_clusters.parquet")

cards_with_clusters = (
    cards_enriched
    .merge(clusters, on="id", how="left")
)

cards_with_clusters[["name", "type_line", "cluster"]].head()

,name,type_line,cluster
0,"Nissa, Worldsoul Speaker",Legendary Creature — Elf Druid,148
1,Mine Security,Creature — Kavu Soldier,122
2,Static Orb,Artifact,91
3,Sensory Deprivation,Enchantment — Aura,54
4,Kavaron Consumed,Sorcery,64


In [29]:
# Inspect a single cluster

cluster_id = 17  # change this

cards_with_clusters[cards_with_clusters["cluster"] == cluster_id][
    ["name", "type_line", "oracle_text_norm"]
].head(20)

,name,type_line,oracle_text_norm
64,Dungeon Delver,Legendary Enchantment — Background,"Commander creatures you own have ""Room abilities of dungeons you own trigger an additional time."""
75,Brawn,Creature — Incarnation,"Trample\nAs long as this card is in your graveyard and you control a Forest, creatures you control have trample."
116,Furnace Oriflamme,Enchantment,"Creatures you control get +1/+0.\nIf you're on the Mirran team, creatures you control have haste.\nIf you're on the Phyrexian team, creatures you control have trample."
1464,Hag of Mage's Doom,Creature — Hag Warlock,"Warlocks you control have ""Ward—Pay 2 life."""
2682,Cyberdrive Awakener,Artifact Creature — Construct,"Flying\nOther artifact creatures you control have flying.\nWhen this creature enters, each noncreature artifact you control becomes a 4/4 artifact creature until end of turn."
2695,"Smellerbee, Rebel Fighter",Legendary Creature — Human Rebel Ally,"First strike\nOther creatures you control have haste.\nWhenever Smellerbee attacks, you may discard your hand. If you do, draw cards equal to the number of attacking creatures."
2711,"Maha, Its Feathers Night",Legendary Creature — Elemental Bird,"Flying, trample\nWard—Discard a card.\nCreatures your opponents control have base toughness 1."
3162,Orim,Vanguard,Creatures you control have reach.
3392,"Elspeth, Knight-Errant Emblem",Emblem — Elspeth,"Artifacts, creatures, enchantments, and lands you control have indestructible."
3657,The Kami Knight,Legendary Creature — Spirit Warrior,Other creatures you control gain all bonuses conferred by equipment attached to this creature.\nCreatures you control can't be goaded.


In [31]:
# Cluster size distribution

cards_with_clusters["cluster"].value_counts().head(20)

cluster
4      3896
61      493
70      431
96      409
99      388
67      387
12      372
68      336
43      326
24      321
22      313
167     309
126     307
2       299
79      292
109     291
195     290
86      290
130     288
23      286
Name: count, dtype: int64